In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/iamrecan/drone-images/FPV drone.v4i.coco/README.dataset.txt
/kaggle/input/datasets/iamrecan/drone-images/FPV drone.v4i.coco/README.roboflow.txt
/kaggle/input/datasets/iamrecan/drone-images/FPV drone.v4i.coco/valid/245_png.rf.074aeaa3c1c4fe4b2f8631f9b1bcb995.jpg
/kaggle/input/datasets/iamrecan/drone-images/FPV drone.v4i.coco/valid/277528076_343035084518475_1026526733656920578_n-1_jpg.rf.8f3ceded577ddbfd9a2495c8235270cb.jpg
/kaggle/input/datasets/iamrecan/drone-images/FPV drone.v4i.coco/valid/1031-Abandoned-Russian-BMP-2-31-10-23_jpg.rf.d54e23a10fb552ff8c97bd18e64c1589.jpg
/kaggle/input/datasets/iamrecan/drone-images/FPV drone.v4i.coco/valid/2005-PT-91-Twardy-dam-aband-26-02-24_jpg.rf.0c7598b372ccb108cb74ab4aa59961bf.jpg
/kaggle/input/datasets/iamrecan/drone-images/FPV drone.v4i.coco/valid/2000-t72m1_jpg.rf.3098fcb0993af7df4501739774993509.jpg
/kaggle/input/datasets/iamrecan/drone-images/FPV drone.v4i.coco/valid/id44379-04_jpg.rf.4a65e97591b9f307e1f30a42b71b2473.jp

In [2]:
# model training 
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.1 MB/s eta 0:00:0000:01


In [ ]:
import os
from pathlib import Path

DATASET_DIR = Path("/kaggle/input/datasets/iamrecan/drone-images/FPV drone.v4i.coco")
MODEL_PATH = Path("/kaggle/input/ultralytics-yolo26-yolo26-v1/yolo26x.pt")  # hızlı başlamak için n
# İstersen sonra bunu yolo26s.pt veya yolo26m.pt yaparsın

print("Dataset exists:", DATASET_DIR.exists())
print("Model exists:", MODEL_PATH.exists())

for split in ["train", "valid", "test"]:
    split_dir = DATASET_DIR / split
    print(f"\n--- {split} ---")
    if split_dir.exists():
        for f in split_dir.iterdir():
            print(f.name)
    else:
        print("Yok")

In [ ]:
import json
import shutil
from pathlib import Path
from collections import defaultdict

SRC_DATASET_DIR = Path("/kaggle/input/datasets/iamrecan/drone-images/FPV drone.v4i.coco")
YOLO_DATASET_DIR = Path("/kaggle/working/drone_yolo_dataset")

def prepare_split(split_name):
    src_split = SRC_DATASET_DIR / split_name
    json_path = src_split / "_annotations.coco.json"
    
    if not src_split.exists() or not json_path.exists():
        print(f"{split_name} yok, geçiliyor.")
        return None

    dst_images = YOLO_DATASET_DIR / "images" / split_name
    dst_labels = YOLO_DATASET_DIR / "labels" / split_name
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)

    with open(json_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    images = {img["id"]: img for img in coco["images"]}
    categories = sorted(coco["categories"], key=lambda x: x["id"])
    cat_id_to_idx = {cat["id"]: i for i, cat in enumerate(categories)}
    class_names = [cat["name"] for cat in categories]

    anns_by_image = defaultdict(list)
    for ann in coco["annotations"]:
        anns_by_image[ann["image_id"]].append(ann)

    # Görselleri kopyala + txt label yaz
    for image_id, img in images.items():
        file_name = img["file_name"]
        w, h = img["width"], img["height"]

        src_img_path = src_split / file_name
        dst_img_path = dst_images / file_name

        if src_img_path.exists():
            shutil.copy2(src_img_path, dst_img_path)
        else:
            print(f"Görsel bulunamadı: {src_img_path}")
            continue

        txt_path = dst_labels / f"{Path(file_name).stem}.txt"
        lines = []

        for ann in anns_by_image.get(image_id, []):
            x, y, bw, bh = ann["bbox"]

            x_center = (x + bw / 2) / w
            y_center = (y + bh / 2) / h
            bw_n = bw / w
            bh_n = bh / h

            cls = cat_id_to_idx[ann["category_id"]]
            lines.append(f"{cls} {x_center} {y_center} {bw_n} {bh_n}")

        with open(txt_path, "w", encoding="utf-8") as f:
            f.write("\n".join(lines))

    print(f"{split_name} hazırlandı.")
    print("Image count:", len(list(dst_images.glob("*"))))
    print("Label count:", len(list(dst_labels.glob("*.txt"))))
    return class_names

all_names = None
for split in ["train", "valid", "test"]:
    names = prepare_split(split)
    if names is not None:
        all_names = names

print("Classes:", all_names)
print("Tamam.")

In [ ]:
import yaml
from pathlib import Path

yaml_path = Path("/kaggle/working/data.yaml")

data_yaml = {
    "path": str(YOLO_DATASET_DIR),
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "names": {i: name for i, name in enumerate(all_names)}
}

with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml, f, allow_unicode=True, sort_keys=False)

print(yaml_path.read_text())

In [ ]:
from pathlib import Path
import shutil
from ultralytics import YOLO

src_model = Path("/kaggle/input/models/ultralytics/yolo26/pytorch/default/1/yolo26x.pt")
dst_model = Path("/kaggle/working/yolo26x.pt")

print("Source exists:", src_model.exists())
shutil.copy2(src_model, dst_model)
print("Copied to:", dst_model)

model = YOLO(str(dst_model))

results = model.train(
    data="/kaggle/working/data.yaml",
    epochs=100,
    imgsz=640,
    batch=8,      # OOM olursa 4 yap
    device=0,
    workers=2,
    project="/kaggle/working",
    name="drone_yolo26_run",
    pretrained=True,
    cache=False,
    patience=10,
    save=True,
    verbose=True
)

In [ ]:
from pathlib import Path
import shutil

run_dir = Path("/kaggle/working/drone_yolo26_run")
best_pt = run_dir / "weights" / "best.pt"

print("Best model:", best_pt)
print("Exists:", best_pt.exists())

shutil.make_archive("/kaggle/working/drone_yolo26_run", "zip", run_dir)
print("ZIP hazır: /kaggle/working/drone_yolo26_run.zip")